# 11주 · 방정식의 근·구간법·개방법

수치해석및실습 CE273

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

학습 주제: 방정식의 근 (1) / 방정식의 근 (2)

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week11/CE273_W11.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week11)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE273-numerical-analysis/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)


## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 계산식 대조


In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def parachute_velocity(t, g=9.81, m=68.1, c=12.5):
    if not (np.isfinite([g,m,c]).all() and g>0 and m>0 and c>=0):
        raise ValueError("g·m 양수, c 비음수 확인 필요")
    t = np.asarray(t, dtype=float)
    if not np.isfinite(t).all() or np.any(t<0):
        raise ValueError("유한한 비음수 시간 필요")
    return g*t if c == 0 else g*m/c*(-np.expm1(-c*t/m))

def parachute_distance(t, g=9.8, m=68.1, c=12.5):
    parachute_velocity(t,g,m,c) # 같은 물리 입력 검증
    t=np.asarray(t,dtype=float)
    if c == 0:
        return .5*g*t*t
    return g*m/c*(t + m/c*np.expm1(-c*t/m))
def root_result(x, history, converged, reason):
    return {"root":float(x),"history":pd.DataFrame(history),"iterations":len(history),"converged":converged,"reason":reason}

def root_options(xtol,ftol,max_iter):
    if not np.isfinite([xtol,ftol]).all() or xtol<=0 or ftol<=0 or not isinstance(max_iter,int) or max_iter<=0:
        raise ValueError("양의 위치·잔차 허용오차, 양의 정수 최대 반복 수 필요")

def bracket_root(f,a,b,method="bisection",xtol=1e-10,ftol=1e-10,max_iter=200):
    root_options(xtol,ftol,max_iter)
    if method not in ["bisection","false_position","illinois"] or not np.isfinite([a,b]).all() or a>=b:raise ValueError("방법·a<b 확인 필요")
    fa,fb=float(f(a)),float(f(b));history=[]
    if not np.isfinite([fa,fb]).all():raise ValueError("유한한 양 끝 함수값 필요")
    if fa==0:return root_result(a,history,True,"left endpoint")
    if fb==0:return root_result(b,history,True,"right endpoint")
    if np.signbit(fa)==np.signbit(fb):raise ValueError("연속성 확인 후 부호 변화 구간 필요")
    wa,wb=fa,fb;previous=None;retained=None;x=(a+b)/2
    for k in range(max_iter):
        x=(a+b)/2 if method=="bisection" else b-wb*(b-a)/(wb-wa)
        fx=float(f(x))
        if not np.isfinite(fx):raise ValueError("반복 중 비유한 함수값")
        step=np.inf if previous is None else abs(x-previous)
        history.append({"iteration":k+1,"a":a,"b":b,"x":x,"residual":abs(fx),"step":step})
        if fx==0 or (abs(fx)<=ftol and (b-a<=2*xtol if method=="bisection" else step<=xtol)):
            return root_result(x,history,True,"position and residual")
        if np.signbit(fa)!=np.signbit(fx):
            b,fb,wb=x,fx,fx
            wa=wa/2 if method=="illinois" and retained=="a" else fa
            retained="a"
        else:
            a,fa,wa=x,fx,fx
            wb=wb/2 if method=="illinois" and retained=="b" else fb
            retained="b"
        previous=x
    return root_result(x,history,False,"maximum iterations")

def open_root(f,x0,method="newton",df=None,g=None,x1=None,xtol=1e-10,ftol=1e-10,max_iter=100):
    root_options(xtol,ftol,max_iter)
    if method not in ["newton","fixed_point","secant"] or not np.isfinite(x0):raise ValueError("방법·초기값 확인 필요")
    if method=="newton" and not callable(df):raise ValueError("도함수 필요")
    if method=="fixed_point" and not callable(g):raise ValueError("고정점 함수 필요")
    if method=="secant" and (x1 is None or not np.isfinite(x1) or x1==x0):raise ValueError("서로 다른 두 유한 초기값 필요")
    previous=float(x0);current=float(x1 if method=="secant" else x0);history=[]
    if float(f(current))==0:return root_result(current,history,True,"initial root")
    for k in range(max_iter):
        fc=float(f(current))
        if method=="fixed_point":nxt=float(g(current))
        elif method=="newton":
            derivative=float(df(current))
            if not np.isfinite(derivative) or abs(derivative)<np.finfo(float).eps:raise ValueError("0 또는 작은 도함수")
            nxt=current-fc/derivative
        else:
            fp=float(f(previous));den=fc-fp
            if not np.isfinite(den) or abs(den)<np.finfo(float).eps:raise ValueError("0 또는 작은 함수값 차이")
            nxt=current-fc*(current-previous)/den
        if not np.isfinite(nxt):raise ValueError("비유한 반복값")
        residual=abs(float(f(nxt)));step=abs(nxt-current)
        if not np.isfinite(residual):raise ValueError("비유한 잔차")
        history.append({"iteration":k+1,"x":nxt,"residual":residual,"step":step})
        if residual==0 or (residual<=ftol and step<=xtol):return root_result(nxt,history,True,"position and residual")
        previous,current=current,nxt
    return root_result(current,history,False,"maximum iterations")


<a id="s01"></a>

## 낙하산 항력계수의 구간법

**먼저 예측**: 부호 변화가 없는 구간에는 근이 절대로 없는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
f=lambda c:float(parachute_velocity(10,9.81,68.1,c))-40
for method in ["bisection","false_position"]:
    result=bracket_root(f,12,16,method)
    print(method,result["root"],result["converged"],result["iterations"])
    assert result["converged"] and abs(f(result["root"]))<1e-8
reference=optimize.brentq(f,12,16)
assert np.isclose(reference,14.801135944991259)
print("독립 라이브러리 기준:",reference)

**조건 변경**: 초기 구간을 바꿔 연속성·부호 조건·반복 상한 확인

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: g9.81·m68.1·t10·v40·c의 kg/s·연속 구간12~16·위치/잔차 허용오차·상한 명시

**결과 검토**: 부호 변화는 연속성 아래 존재 보장, 유일성의 일반 보장 아님

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


<a id="s02"></a>

## 고정점·Newton의 수렴

**먼저 예측**: 근은 같아도 반복식과 초기값에 따라 수렴이 달라지는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
f=lambda x:np.exp(-x)-x
for method in ["fixed_point","newton"]:
    out=open_root(f,0,method,df=lambda x:-np.exp(-x)-1,g=lambda x:np.exp(-x))
    print(method,out["root"],out["iterations"],out["converged"])
    assert out["converged"] and np.isclose(out["root"],.5671432904097838)
first=np.exp(-0.0);assert first==1
print("원본 Excel B4는 반복번호 A3 참조. 초기값 변경 시 이전 x B3 참조가 필요")

**조건 변경**: 고정점식 exp(-x)와 다른 반복식을 비교. 발산 시 성공으로 보고하지 않음

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: f·도함수/고정점식·x0·잔차·갱신량·최대 반복 수·실패 상태 명시

**결과 검토**: 초기값 변경 시 반복표의 직전 행 참조와 첫 갱신을 확인

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


<a id="s03"></a>

## 할선법의 두 초기값

**먼저 예측**: 할선법의 두 초기값은 항상 근을 끼워야 하는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
f=lambda x:np.exp(-x)-x
out=open_root(f,0,"secant",x1=1)
print(out["history"])
assert out["converged"] and np.isclose(out["root"],.5671432904097838)
limited=open_root(f,0,"newton",df=lambda x:-np.exp(-x)-1,max_iter=1)
assert not limited["converged"]
print("반복 상한 도달:",limited["reason"])

**조건 변경**: 같은 초기값 두 개를 넣고 함수값 차이 분모의 실패 조건 확인

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 개방법·두 초기값0/1·분모 점검·동일한 위치/잔차 기준·실패 상태 명시

**결과 검토**: Newton·할선법의 수렴은 무조건 보장되지 않음

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

모형과 단위·수렴·실패 상태·독립 기준값 검토

입력 항목:

- `항력계수_c_kg_s`: 해당 계산값·항 수·오차 기준의 수치
- `exp마이너스x의_근`: 해당 계산값·항 수·오차 기준의 수치
- `상한_도달값은_항상_수렴해`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인


In [ ]:
answers = {'항력계수_c_kg_s': None, 'exp마이너스x의_근': None, '상한_도달값은_항상_수렴해': None}


In [ ]:
def check_answers(values):
    expected = {'항력계수_c_kg_s': 14.801135944991259, 'exp마이너스x의_근': 0.5671432904097838, '상한_도달값은_항상_수렴해': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음


In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
